In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping
import pickle

In [12]:
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# Encoding the gender using Label Encoder
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# One-Hot Encoding Geography
onehot_encode_geo = OneHotEncoder()
geo_encoded = onehot_encode_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns = onehot_encode_geo.get_feature_names_out(['Geography']))

# Concatinating the data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X = data.drop('Exited',axis=1)
y = data['Exited']

In [14]:
# Performing Train Test Split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [16]:
# Saving the encoded sets and scaled sets
with open('label_encoder_gender_hpt.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encode_geo_hpt.pkl','wb') as file:
    pickle.dump(onehot_encode_geo,file)

with open('scaler_hpt.pkl','wb') as file:
    pickle.dump(scaler,file)

In [52]:
# Define a function to create a model and try different models --> use keras classifier

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam',loss = 'binary_crossentropy', metrics=['accuracy'])

    return model


In [53]:
# Create a Keras Classifier

model = KerasClassifier(layers=1,neurons=32,model=create_model,epochs=50, batch_size=10, verbose=0)


In [54]:
# Defining the Grid Search parameters
param_grid = {
    'neurons': [16,32,64,128],
    'layers' : [1,2],
    #'batch_size' : [10,20],
    'epochs' : [50,100]
}


In [55]:
# Performing Grid Search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=2)

print("Starting Exhaustive Grid Search on CPU...")
grid_result = grid.fit(X_train,y_train)

print("\n=== GRID SEARCH COMPLETE ===")
# Printing the best parameters
#print("Best : %f using %s" % (grid_result.best_score_, grid_result.best_params_))
print("Best Score: %f" % grid_result.best_score_)
print("Best Parameter Combination Found:")
for param, value in grid_result.best_params_.items():
    print(f" -> {param}: {value}")

Starting Exhaustive Grid Search on CPU...
Fitting 3 folds for each of 16 candidates, totalling 48 fits


d:\GenAICourse\ANN-BinaryClassification\annenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== GRID SEARCH COMPLETE ===
Best Score: 0.856999
Best Parameter Combination Found:
 -> epochs: 50
 -> layers: 1
 -> neurons: 32
